In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

glof = pd.read_csv("../build_ml_GLOF.csv")
glof.head()

,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,area_change_km2,pct_of_critical_area,yrs_to_critical_area,expansion_accel_km2yr2,dam_stability_index,peak_discharge_m3s
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,0.000000,1.00,500.0,0.000000,0.0367,121.69
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,0.000000,11.12,500.0,0.000000,0.0657,722.82
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,-0.003574,10.41,500.0,0.000000,0.0660,684.41
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,0.000249,11.17,500.0,0.007398,0.0968,725.49
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,-0.000831,10.96,500.0,-0.004904,0.0824,713.93


### Cleaning

In [2]:
glof = glof.sort_values(["lake_id", "year"]).reset_index(drop=True)
glof.head()

,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,area_change_km2,pct_of_critical_area,yrs_to_critical_area,expansion_accel_km2yr2,dam_stability_index,peak_discharge_m3s
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,0.000000,1.00,500.0,0.000000,0.0367,121.69
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,0.000000,11.12,500.0,0.000000,0.0657,722.82
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,-0.003574,10.41,500.0,0.000000,0.0660,684.41
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,0.000249,11.17,500.0,0.007398,0.0968,725.49
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,-0.000831,10.96,500.0,-0.004904,0.0824,713.93


#### calculating the latitude and longitude from the data
#### this will come in handy later on, when we have to see info on individual lakes

In [3]:
parts = glof["lake_id"].str.split("_", expand=True)
glof["lon"] = parts[1].astype(float)
glof["lat"] = parts[2].astype(float)
glof[["lake_id", "lon", "lat"]].head()

,lake_id,lon,lat
0,GLO_80.85718_29.91975,80.85718,29.91975
1,GLO_80.87763_30.04078,80.87763,30.04078
2,GLO_80.87763_30.04078,80.87763,30.04078
3,GLO_80.87763_30.04078,80.87763,30.04078
4,GLO_80.87763_30.04078,80.87763,30.04078


### Cleaning the new columns

In [4]:
# - 500 means "not in the forecast window", not a real 500-year wait
# - negative years means the lake is already past the 0.005 km2 critical area
# - pct_of_critical_area is area / 0.005, so we do not train on it

glof["yrs_capped"] = (glof["yrs_to_critical_area"] == 500).astype(int)
glof["already_past_critical"] = (glof["yrs_to_critical_area"] < 0).astype(int)
glof["glof_within_50yr"] = (glof["yrs_to_critical_area"] < 50).astype(int)
print(glof["glof_within_50yr"].value_counts())
print("missing:", glof.isna().sum().sum())

glof_within_50yr
0    11677
1      227
Name: count, dtype: int64
missing: 0


### Engineered features

In [5]:
# - mean_depth: volume relative to area
# - uncertainty_frac: measurement noise relative to lake size
# - expansion_frac: growth relative to current area
# - volume_gap: stored volume vs a simple area*height box
# - volume_dam left out of the model later (formula costume)

glof["mean_depth"] = glof["lake_volume_1e6_m3"] / glof["lake_area_km2"]
glof["volume_dam"] = glof["lake_volume_1e6_m3"] * glof["dam_height_m"]
glof["volume_gap"] = glof["lake_volume_1e6_m3"] - glof["lake_area_km2"] * glof["dam_height_m"]
glof["uncertainty_frac"] = glof["area_uncertainty_km2"] / glof["lake_area_km2"]
glof["expansion_frac"] = glof["expansion_rate_km2yr"] / glof["lake_area_km2"]

glof[["mean_depth", "volume_gap", "uncertainty_frac", "expansion_frac"]].head()

,mean_depth,volume_gap,uncertainty_frac,expansion_frac
0,12.209302,-0.013920,0.601443,0.0
1,20.008631,0.223480,0.179821,0.0
2,19.718662,0.202354,0.192171,0.0
3,20.028643,0.225040,0.179019,0.0
4,19.943410,0.218759,0.182548,0.0


## Fixing leakage

- do not use `pct_of_critical_area` or `yrs_to_critical_area` as features
- do not use `peak_discharge_m3s` (old formula target)
- split by `lake_id`, not by row
- `area_change` is 0 on a lake's first year — leave it out


In [6]:
from sklearn.model_selection import train_test_split

idx_tr, idx_te = train_test_split(glof.index, test_size=0.25, random_state=42)
overlap = set(glof.loc[idx_tr, "lake_id"]) & set(glof.loc[idx_te, "lake_id"])
print("random split, lakes on both sides:", len(overlap))

random split, lakes on both sides: 1438


In [7]:
# splitting by lake

lakes = glof["lake_id"].drop_duplicates().to_numpy()
np.random.seed(42)
np.random.shuffle(lakes)
cut = int(0.75 * len(lakes))
train_lakes = lakes[:cut]
test_lakes = lakes[cut:]

train = glof[glof["lake_id"].isin(train_lakes)].copy()
test = glof[glof["lake_id"].isin(test_lakes)].copy()

print("train rows", len(train), "test rows", len(test))
print("train lakes", train["lake_id"].nunique(), "test lakes", test["lake_id"].nunique())
print("lakes on both sides:", len(set(train["lake_id"]) & set(test["lake_id"])))
print("positive share train", train["glof_within_50yr"].mean(), "test", test["glof_within_50yr"].mean())

train rows 8799 test rows 3105
train lakes 1746 test lakes 583
lakes on both sides: 0
positive share train 0.014660756904193658 test 0.031561996779388084


In [9]:
FEATURES = [
    "year",
    "lake_area_km2",
    "elevation_m",
    "perimeter_km",
    "area_uncertainty_km2",
    "expansion_rate_km2yr",
    "lake_volume_1e6_m3",
    "dam_height_m",
    "shape_index",
    "expansion_accel_km2yr2",
    "dam_stability_index",
    "lon",
    "lat",
    "mean_depth",
    "volume_gap",
    "uncertainty_frac",
    "expansion_frac",
]
TARGET = "glof_within_50yr"

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print("target in features:", TARGET in FEATURES)
print("pct in features:", "pct_of_critical_area" in FEATURES)
print("yrs in features:", "yrs_to_critical_area" in FEATURES)
print(X_train.shape, X_test.shape)
X_train.head()

target in features: False
pct in features: False
yrs in features: False
(8799, 17) (3105, 17)


,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,expansion_accel_km2yr2,dam_stability_index,lon,lat,mean_depth,volume_gap,uncertainty_frac,expansion_frac
0,2024,0.004988,4782.9,0.36469,0.003,0.0000,0.0609,15.0,1.4567,0.000000,0.0367,80.85718,29.91975,12.209302,-0.013920,0.601443,0.000000
5,2017,0.030424,4371.9,0.91173,0.007,0.0005,0.5285,15.0,1.4745,0.000000,0.0508,80.88324,30.05350,17.371154,0.072140,0.230082,0.016434
6,2018,0.031837,4371.9,1.00291,0.008,0.0005,0.5586,15.0,1.5856,0.000000,0.0598,80.88324,30.05350,17.545623,0.081045,0.251280,0.015705
7,2019,0.032419,4371.9,1.03938,0.008,0.0005,0.5711,15.0,1.6284,-0.000831,0.0667,80.88324,30.05350,17.616213,0.084815,0.246769,0.015423
8,2020,0.035412,4371.9,1.13055,0.008,0.0005,0.6360,15.0,1.6948,0.002411,0.0795,80.88324,30.05350,17.960014,0.104820,0.225912,0.014120


In [10]:
glof.to_csv("glof_clean.csv", index=False)
print(glof.shape)
glof.head()

(11904, 26)


,lake_id,year,lake_area_km2,elevation_m,perimeter_km,area_uncertainty_km2,expansion_rate_km2yr,lake_volume_1e6_m3,dam_height_m,shape_index,...,lon,lat,yrs_capped,already_past_critical,glof_within_50yr,mean_depth,volume_dam,volume_gap,uncertainty_frac,expansion_frac
0,GLO_80.85718_29.91975,2024,0.004988,4782.9,0.36469,0.003,0.0,0.0609,15.00,1.4567,...,80.85718,29.91975,1,0,0,12.209302,0.913500,-0.013920,0.601443,0.0
1,GLO_80.87763_30.04078,2017,0.055611,4448.8,1.25819,0.010,0.0,1.1127,15.99,1.5051,...,80.87763,30.04078,1,0,0,20.008631,17.792073,0.223480,0.179821,0.0
2,GLO_80.87763_30.04078,2018,0.052037,4448.8,1.23996,0.010,0.0,1.0261,15.83,1.5334,...,80.87763,30.04078,1,0,0,19.718662,16.243163,0.202354,0.192171,0.0
3,GLO_80.87763_30.04078,2019,0.055860,4448.8,1.27643,0.010,0.0,1.1188,16.00,1.5235,...,80.87763,30.04078,1,0,0,20.028643,17.900800,0.225040,0.179019,0.0
4,GLO_80.87763_30.04078,2020,0.054780,4448.8,1.22172,0.010,0.0,1.0925,15.95,1.4725,...,80.87763,30.04078,1,0,0,19.943410,17.425375,0.218759,0.182548,0.0
